# 📡 Módulo 15 — Workflows: Eventos y Patrones Avanzados

> **Objetivo:** Observar el progreso de un workflow en tiempo real y construir loops convergentes.

## Eventos durante la ejecución

Con `stream=True` podemos iterar sobre los eventos que emite el workflow:

| Evento | Cuándo |
|--------|--------|
| `executor_invoked` | Un ejecutor comenzó a procesar un mensaje |
| `executor_completed` | Un ejecutor terminó (con el resultado en `event.data`) |
| `output` | Un ejecutor llamó a `ctx.yield_output(...)` |

## Patrón loop (bucle convergente)

Dos ejecutores se envían mensajes mutuamente hasta cumplir una condición:

```
[Guesser] ──guess(int)──→ [Judge]
   ↑                          │
   └──feedback(signal)────────┘   (until correct)
```

Cuando el judge acierta, llama a `yield_output(...)` y el loop termina.

> 🎉 **No requiere Azure OpenAI.**


In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from enum import Enum
from typing_extensions import Never
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, executor, handler

print("✅ Listo")


## 1️⃣ Capturar eventos `executor_completed`

`workflow.run(input, stream=True)` devuelve un async iterator de eventos.
Filtramos los `executor_completed` para ver el progreso paso a paso.


In [ ]:
@executor(id="uppercase")
async def uppercase(text: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(text.upper())


@executor(id="doubler")
async def doubler(text: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(text + text)


@executor(id="suffix_output")
async def suffix_output(text: str, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(text + "_DONE")


workflow = (
    WorkflowBuilder(start_executor=uppercase)
    .add_edge(uppercase, doubler)
    .add_edge(doubler, suffix_output)
    .build()
)

completed_count = 0
output_count = 0

print("📡 Eventos en tiempo real:\n")
async for event in workflow.run("hi", stream=True):
    evt_type = getattr(event, "type", None)
    if evt_type == "executor_completed":
        completed_count += 1
        print(f"   ✔ executor_completed: {getattr(event, 'executor_id', '?')} → {event.data}")
    elif evt_type == "output":
        output_count += 1
        print(f"   🎯 output: {event.data}")

print(f"\n✅ Total: {completed_count} pasos, {output_count} outputs")


## 2️⃣ Capturar el evento `output` específicamente

Si sólo te interesa la salida final, filtra por `event.type == "output"`.


In [ ]:
workflow2 = (
    WorkflowBuilder(start_executor=uppercase)
    .add_edge(uppercase, suffix_output)
    .build()
)

outputs: list[str] = []
async for event in workflow2.run("test", stream=True):
    if getattr(event, "type", None) == "output":
        outputs.append(str(event.data))
        print(f"   🎯 WorkflowOutput: {event.data}")

print(f"\n✅ Outputs finales: {outputs}")


## 3️⃣ Loop convergente — adivinar un número

Patrón clásico de **búsqueda binaria** implementado con dos ejecutores conectados bidireccionalmente:
- `Guesser` propone un número (midpoint del rango actual)
- `Judge` evalúa: si acierta → output final, si no → envía `TOO_HIGH` / `TOO_LOW` al Guesser

El loop converge en ~7 iteraciones para un rango de 1-100.


In [ ]:
class GuessSignal(str, Enum):
    INIT = "INIT"
    TOO_HIGH = "TOO_HIGH"
    TOO_LOW = "TOO_LOW"


class GuesserExecutor(Executor):
    def __init__(self, low: int, high: int, id: str = "guesser"):
        super().__init__(id=id)
        self._low, self._high = low, high

    @property
    def midpoint(self) -> int:
        return (self._low + self._high) // 2

    @handler
    async def on_signal(self, signal: GuessSignal, ctx: WorkflowContext[int]) -> None:
        if signal == GuessSignal.TOO_HIGH:
            self._high = self.midpoint - 1
        elif signal == GuessSignal.TOO_LOW:
            self._low = self.midpoint + 1
        await ctx.send_message(self.midpoint)


class JudgeNumberExecutor(Executor):
    def __init__(self, target: int, id: str = "judge"):
        super().__init__(id=id)
        self._target = target
        self._attempts = 0

    @handler
    async def on_guess(
        self, guess: int, ctx: WorkflowContext[GuessSignal, str]
    ) -> None:
        self._attempts += 1
        if guess == self._target:
            await ctx.yield_output(f"Found {self._target} in {self._attempts} attempts!")
        elif guess > self._target:
            await ctx.send_message(GuessSignal.TOO_HIGH)
        else:
            await ctx.send_message(GuessSignal.TOO_LOW)


target = 42
guesser = GuesserExecutor(low=1, high=100)
judge = JudgeNumberExecutor(target=target)

workflow = (
    WorkflowBuilder(start_executor=guesser)
    .add_edge(guesser, judge)   # guess (int) → judge
    .add_edge(judge, guesser)   # signal (GuessSignal) → guesser
    .build()
)

events = await workflow.run(GuessSignal.INIT)
print(f"🎯 Resultado del loop: {events.get_outputs()}")


## 🎯 Resumen

| Capacidad | Cómo se hace |
|-----------|--------------|
| Iterar eventos | `async for event in workflow.run(input, stream=True):` |
| Filtrar por tipo | `event.type in ("executor_invoked", "executor_completed", "output")` |
| Loop convergente | Conexión bidireccional + un nodo que llama `yield_output` para terminar |

> 💡 Los loops son útiles para: **refinamiento iterativo**, **búsqueda con feedback**,
> **retry con backoff**, **convergencia matemática** (Newton-Raphson, etc.).

**Siguiente módulo →** [Módulo 16 — Múltiples Agentes Coordinados](./16_multiple_agents.ipynb)
